# DataSphere JupyterLab setup

This notebook installs the repository into the current Jupyter kernel, checks CUDA, and helps update the Git checkout. It does not launch DataSphere Jobs or download models.

In [ ]:
from pathlib import Path
import os
import sys

def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Open this notebook from inside the cloned repository.")

REPOSITORY_ROOT = find_repository_root(Path.cwd())
os.chdir(REPOSITORY_ROOT)
print("Repository:", REPOSITORY_ROOT)
print("Kernel Python:", sys.executable)
print("Python version:", sys.version)

## Install the environment

Set `RUN_INSTALLATION = True`. Enable the GPU layer only on a GPU notebook VM when you need the complete RL/vLLM environment. The first GPU installation can take around ten minutes; rerunning it on the same VM should reuse installed packages.

In [ ]:
import subprocess

RUN_INSTALLATION = False
INSTALL_DEV_TOOLS = True
INSTALL_GPU_DEPENDENCIES = False

command = [sys.executable, "scripts/setup_jupyter.py"]
if INSTALL_DEV_TOOLS:
    command.append("--dev")
if INSTALL_GPU_DEPENDENCIES:
    command.append("--gpu")

print("Selected command:", subprocess.list2cmdline(command))
if RUN_INSTALLATION:
    subprocess.run(command, check=True)
else:
    subprocess.run([*command, "--dry-run"], check=True)
    print("Dry run only. Set RUN_INSTALLATION = True and rerun this cell to install.")

After installation, select **Kernel → Restart Kernel**, then rerun the first cell and continue below.

In [ ]:
import json
from verifier_bottleneck.cli import collect_system_report

system_report = collect_system_report()
print(json.dumps(system_report, indent=2, sort_keys=True))

In [ ]:
REQUIRE_CUDA = False

if REQUIRE_CUDA and not system_report["cuda_available"]:
    raise RuntimeError("CUDA is required, but this notebook kernel cannot access it.")
print("Environment smoke check passed.")

## Optional full dependency import check

Run this only after installing the GPU dependency layer. It imports every package listed in the current full GPU environment and writes a JSON report.

In [ ]:
RUN_FULL_DEPENDENCY_CHECK = False

if RUN_FULL_DEPENDENCY_CHECK:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "verifier_bottleneck.dependency_report",
            "--output",
            "jupyter-dependency-report.json",
            "--require-cuda",
        ],
        check=True,
    )
else:
    print("Skipped. Set RUN_FULL_DEPENDENCY_CHECK = True when needed.")

## Update the repository

Commit or save notebook changes before pulling. The pull uses `--ff-only`, so it will stop rather than create an unexpected merge.

In [ ]:
PULL_LATEST = False

subprocess.run(["git", "status", "--short"], check=True)
if PULL_LATEST:
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    print("Pull skipped. Set PULL_LATEST = True after reviewing git status.")

When finished, save outputs you need and release the notebook VM from the DataSphere interface. Closing the browser tab does not necessarily stop billing.